# 2. Export and interchange

One model, four serializations:

| Format | Functions | Notes |
|---|---|---|
| SysML v2 text | `to_sysml` | re-parseable; round-trips preserve the model |
| JSON | `to_json` / `from_json` | lossless; also the cache format (no pickles) |
| KerML | `to_kerml` | one-way projection onto the kernel language |
| OMG API JSON | `sysml2.api` | flat `@type`/`@id` records (needs pyecore) |

In [ ]:
import sysml2

model = sysml2.loads('''
package Demo {
    part def Battery { attribute capacity : Real = 5200.0; }
    part def Drone {
        attribute mass : Real = 1.2;
        part battery : Battery;
    }
    calc def HoverTime { in c : Real; return : Real = c / 12000.0 * 60.0; }
}
''')
print(sysml2.to_sysml(model))

## JSON is lossless: parse it back and keep executing

In [ ]:
json_text = sysml2.to_json(model)
clone = sysml2.from_json(json_text)

assert sysml2.to_dict(clone) == sysml2.to_dict(model)
print("round-trip preserved the model bit-for-bit")
print("HoverTime from the clone:",
      sysml2.Interpreter(clone).call("Demo::HoverTime", c=5200.0))

## `save()` / `load()` dispatch on the file suffix

In [ ]:
import tempfile
from pathlib import Path

out = Path(tempfile.mkdtemp())
for suffix in (".sysml", ".json", ".kerml"):
    sysml2.save(model, out / f"demo{suffix}")
sorted(p.name for p in out.iterdir())

## KerML projection

SysML v2 is defined as an extension of KerML: `part def` -> `struct`,
`calc def` -> `function`, constraints -> `inv`, ... The projection is
one-way but guaranteed re-parseable by the bundled KerML grammar.

In [ ]:
kerml_text = sysml2.to_kerml(model)
print(kerml_text)
sysml2.parse_kerml_text(kerml_text)   # raises on invalid KerML
print("KerML output re-parses cleanly")

## OMG spec metamodel and API JSON (optional, `pip install sysml2[ecore]`)

`sysml2.ecore` projects models onto the *specification* abstract syntax
(the pilot implementation's `SysML.ecore`, 175 metaclasses) with reified
memberships and relationship elements; `sysml2.api` emits the flat records
SysML v2 tools exchange.

In [ ]:
try:
    from sysml2 import api, ecore
except ImportError:
    print("pyecore not installed -- skipping")
else:
    spec = ecore.to_spec(model)
    print("projection report:", spec.report)

    records = api.to_api_records(model)
    part_def = next(r for r in records if r.get("declaredName") == "Drone")
    print("\nAPI record for Drone:")
    for key, value in list(part_def.items())[:5]:
        print(f"  {key}: {value}")